# 🎛️ console_explore — the operator console engine (M6.4)

The console you drive at `just console` is a thin SPA over one Python object: `Console`
in `e2e/dashboard/orchestrator.py`. This notebook builds that object by hand and calls it,
so you see the *real* pipeline the browser only visualizes — every A2A message, MCP tool
call, on-chain tx, predicate check, and device config, as the typed events the SPA renders.

Nothing here is mocked: real Anvil, real EIP-712, real ERC-721, real gNMI writes to srl1.

**Prerequisites:** the lab is up (`containerlab deploy -t netlab/topology.clab.yml`) for the
live router lane. **Companions:** [`docs/08-demo-dashboard.md`](../../docs/08-demo-dashboard.md) ·
`e2e/dashboard/orchestrator.py` (this engine) · `e2e/dashboard/server.py` (the SSE wrapper) ·
[`docs/adr/007-telemetry-delivery.md`](../../docs/adr/007-telemetry-delivery.md).

## 1. One `Console` = the whole stack, wired

`Console.ensure_started()` launches Anvil, deploys the contracts, seeds 6 presales (so Ada's
first ticket is literally #7), builds Ada's & Bell's chainmcp clients, a real controller, and
— if the lab is up — a `GnmiProvisioner` for srl1. We pass a `show` callback that prints each
event the way the browser would render it.

In [1]:
from e2e.dashboard.orchestrator import Console

def show(ev):
    k = ev["kind"]
    if k == "stage":
        print(f"\n▸ [{ev['domain']:10}] {ev['title']}")
    elif k == "boot":
        print(f"  boot      {ev['title']} — {ev['detail']}")
    else:
        print(f"    {k:10} {ev.get('title','')}  —  {ev.get('detail','')[:88]}")

console = Console()
print("lab detected:", console.status()["lab"])

lab detected: True


## 2. Buy bandwidth — watch it cross the four trust domains

This is exactly what the chat "get me 50 Mbps" does. Read the printout as the trust relay:
**agent** negotiates + judges → **chain** settles (real tx hash, ERC-721 #7) → **controller**
authorizes (predicate 6/6) → **network** writes a real policer to srl1 and iperf measures it.

In [2]:
console.provision("bandwidth", budget_tok=15, emit=show)

  boot      Starting local chain — anvil — story time 13:32
  boot      Contracts deployed — MockTOK 0x5FbDB231…  ·  A2ASettlement 0xe7f1725E…


  boot      Router online — SR Linux srl1 @ 172.20.20.4
  boot      Controller ready — predicate · auth · session machine

▸ [agent     ] Agents negotiate
    a2a        Ada → Bell  —  quote_bandwidth · need bandwidth · 50 Mbps · window 14:00–16:00
    mcp        Bell · chainmcp.sign_offer  —  bandwidth @ 10 TOK → EIP-712 signature (65 bytes)
    a2a        Bell → Ada  —  signed_offer · 10 TOK · signed
    decision   Ada decides: accept  —  10 TOK ≤ budget 15 TOK — accept

▸ [chain     ] Payment for ticket, atomically
    mcp        Ada · chainmcp.fulfill_offer  —  approve 10 TOK + fulfill → …
    chain_tx   tx · fulfill  —  ticket #7 → Ada · 10 TOK → Bell · salt consumed  ·  0x6d4bb60fe4a27775…  block 18
    chain_read ownerOf(7)  —  0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
    chain_read Bell balance  —  70 TOK

▸ [controller] Authorize by on-chain ownership
    mcp        controller · ctrl-mcp.get_challenge  —  entitlement #7 → nonce 0x65acf1474236…
    mcp        Ada · chainmcp.s

    device     srl1 config  —  policer a2a-ent7-a0 · 50 Mbps · oper up


    iperf      iperf3 hostA→hostB  —  offered 100 Mbps → 49 Mbps received (policed)
    done       done  —  bandwidth live — ticket #7. Bell can revoke it any time.


## 3. Inspect srl1 — the enforcement, read live off the router

`device_state()` is the same gNMI read the Device Inspector panel shows: the policer named
after the session and the interface oper-state, straight from srl1.

In [3]:
import json
print(json.dumps(console.device_state(), indent=2))

{
  "online": true,
  "interface": "ethernet-1/1",
  "oper": "up",
  "policers": [
    {
      "name": "a2a-ent7-a0",
      "peak_kbps": 50000
    }
  ]
}


## 4. Bell revokes — the signal is cut at the chain

One on-chain `revoke()`. The controller's watcher tears the session down, the policer is
removed from srl1, and iperf returns to the ~100 Mbps ceiling. The throughput died because
an ERC-721 flag flipped on a blockchain — nobody touched the router directly.

In [4]:
console.revoke(emit=show)


▸ [chain     ] Bell pulls the ticket
    chain_tx   tx · revoke  —  Revoked(#7) — flag flips on-chain  ·  0xba6cb8977fa53632…  block 20

▸ [controller] Watcher tears the session down
    teardown   Session ent7-a0 torn down  —  revoked ticket → no authorization

▸ [network   ] Enforcement withdrawn


    device     srl1 config  —  no policer · oper up — config read live off the router


    iperf      iperf3 hostA→hostB  —  offered 100 Mbps → 100 Mbps received (unshaped)
    done       done  —  Ticket #7 revoked. Throughput returned to unshaped.


## 5. The *other* product: telemetry = the RIGHT to configure the device (ADR-007)

Symmetric with bandwidth. Buying telemetry authorizes the controller to write a real gNMI
**dial-out export destination** (`grpc-tunnel`) to srl1 pointing at Ada's collector. The
token *is* the access. Watch the `device` event, then read the config back off the router.

In [5]:
console.provision("telemetry", budget_tok=15, emit=show)
print("\ngrpc-tunnel destinations on srl1:", console.provisioner.telemetry_config("srl1"))


▸ [agent     ] Agents negotiate
    a2a        Ada → Bell  —  quote_telemetry · need telemetry · 10 s samples · window 14:00–16:00
    mcp        Bell · chainmcp.sign_offer  —  telemetry @ 8 TOK → EIP-712 signature (65 bytes)
    a2a        Bell → Ada  —  signed_offer · 8 TOK · signed
    decision   Ada decides: accept  —  8 TOK ≤ budget 15 TOK — accept

▸ [chain     ] Payment for ticket, atomically
    mcp        Ada · chainmcp.fulfill_offer  —  approve 8 TOK + fulfill → …
    chain_tx   tx · fulfill  —  ticket #8 → Ada · 8 TOK → Bell · salt consumed  ·  0xb5f6dfe9e0f51285…  block 22
    chain_read ownerOf(8)  —  0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
    chain_read Bell balance  —  78 TOK

▸ [controller] Authorize by on-chain ownership
    mcp        controller · ctrl-mcp.get_challenge  —  entitlement #8 → nonce 0x63efbb46bcce…
    mcp        Ada · chainmcp.sign_activation_proof  —  a2a-activate|… → EIP-191 signature
    predicate  predicate 6/6  —  owner✓ · started✓ · not-expir

## 6. The chat front door — plain language → what to buy

The console's chat routes a sentence to one product with `_parse_intent` (deterministic by
default for speed; a real LLM behind `A2A_CHAT_LLM=1`). No stack needed — it's pure parsing:

In [6]:
from e2e.dashboard.orchestrator import _parse_intent

for msg in [
    "get me 50 Mbps from hostA to hostB, budget 12 TOK",
    "I want the right to configure telemetry export on srl1",
    "monitor the interface counters",
    "buy bandwidth, I can spend 8 tokens",
]:
    i = _parse_intent(msg)
    print(f"{msg!r}\n   → {i['service']:9} budget {i['budget']:>2} TOK  —  {i['label']}\n")

'get me 50 Mbps from hostA to hostB, budget 12 TOK'
   → bandwidth budget 12 TOK  —  a 50 Mbps bandwidth guarantee

'I want the right to configure telemetry export on srl1'
   → telemetry budget 15 TOK  —  the right to configure telemetry export on srl1

'monitor the interface counters'
   → telemetry budget 15 TOK  —  the right to configure telemetry export on srl1

'buy bandwidth, I can spend 8 tokens'
   → bandwidth budget  8 TOK  —  a 50 Mbps bandwidth guarantee



## What you learned (and where it goes)

| You did | The concept | In the SPA |
|---|---|---|
| `provision("bandwidth")` with a print emitter | the real pipeline emits typed events | the trust relay + event stream |
| `device_state()` | enforcement read live off srl1 | the Device Inspector |
| `revoke()` | on-chain flag → controller watcher → teardown | the "cut" animation + gauge return |
| `provision("telemetry")` → `telemetry_config()` | telemetry = a device-config *right* (ADR-007) | the telemetry inspector |
| `_parse_intent(...)` | plain language → one product | "Talk to Ada's agent" |

## Experiments to try

- Call `console.provision("bandwidth", budget_tok=5, emit=show)` — Ada declines (5 < 10 TOK).
  Where in the printout does the flow stop, and why did nothing hit the chain?
- After a bandwidth provision, run `console.device_state()` then `console.revoke(show)` then
  `console.device_state()` again — diff the policer list.
- Flip `A2A_CHAT_LLM=1` in the environment (needs a running LLM) and re-run cell 6 — now the
  *reasoning* comes from the model, not the keyword parser.
- Read `server.py`: how do these same events reach the browser? (one SSE queue per client.)

In [7]:
console.close()  # stop Anvil, drop gNMI connections, remove any leftover config
print("stack down.")

stack down.
